### <div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 15px; border-radius: 4px; color: #0c5460;"> <strong>Imports</strong>
</div>.

In [1]:
import sys, os
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import BATCH_SIZE, DEVICE, SAMPLE_FRAC, EPOCHS, CLASS_NAMES, RANDOM_SEED, UNFREEZE_EPOCH, \
LEARNING_RATE, MEL_CLASS_INDEX, MEL_CLASS_WEIGHT_MULTIPLIER
from src.dataset import SkinCancerDataset, get_transforms
from src.model import build_model

import time
start_time = time.time()

In [2]:
datadir = PROJECT_ROOT / "data/raw"
metadata_path = datadir / "HAM10000_metadata.csv"

df = pd.read_csv(metadata_path)

image_dir_1 = datadir / "HAM10000_images_part_1"
image_dir_2 = datadir / "HAM10000_images_part_2"

image_path_dict = {
    os.path.splitext(f)[0]: str((image_dir_1 / f).relative_to(PROJECT_ROOT))
    for f in os.listdir(image_dir_1)
}

image_path_dict.update({
    os.path.splitext(f)[0]: str((image_dir_2 / f).relative_to(PROJECT_ROOT))
    for f in os.listdir(image_dir_2)
})

df["path"] = df["image_id"].map(image_path_dict)

In [3]:
torch.cuda.is_available()
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 4090'

In [4]:
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,data/raw/HAM10000_images_part_1/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,data/raw/HAM10000_images_part_1/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,data/raw/HAM10000_images_part_1/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,data/raw/HAM10000_images_part_1/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,data/raw/HAM10000_images_part_2/ISIC_0031633.jpg


In [5]:
df.columns

Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'path'],
      dtype='object')

In [6]:
dataset = SkinCancerDataset(
    df=df,
    transform=get_transforms(),
    sample_frac=SAMPLE_FRAC,
    image_path_col="path"
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

images, labels = next(iter(loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

Image batch shape: torch.Size([64, 3, 224, 224])
Label batch shape: torch.Size([64])
Labels: tensor([5, 5, 5, 4, 1, 5, 5, 5, 4, 5])


In [7]:
print(images.shape)

torch.Size([64, 3, 224, 224])


In [9]:
# Label mapping
label_map = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "df": 3,
    "mel": 4,
    "nv": 5,
    "vasc": 6
}
df["label"] = df["dx"].map(label_map)
# Train / validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=RANDOM_SEED
)
# Datasets
train_dataset = SkinCancerDataset(
    df=train_df,
    transform=get_transforms(train=True),
    sample_frac=SAMPLE_FRAC,
    image_path_col="path"
)
val_dataset = SkinCancerDataset(
    df=val_df,
    transform=get_transforms(train=False),
    sample_frac=1.0,
    image_path_col="path"
)
# Weighted sampler for class imbalance
train_labels = train_dataset.df["label"].values
class_counts = np.bincount(train_labels)
class_sample_weights = 1.0 / class_counts
sample_weights = class_sample_weights[train_labels]
sample_weights = torch.tensor(sample_weights, dtype=torch.float)
weighted_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
# Model, loss, optimizer
model = build_model().to(DEVICE)

# Class-weighted loss with melanoma boost
class_loss_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(train_dataset.df["label"].unique())),
    y=train_dataset.df["label"].values
)
class_loss_weights = torch.tensor(class_loss_weights, dtype=torch.float)
class_loss_weights = torch.clamp(class_loss_weights, min=0.5, max=2.0)
class_loss_weights[MEL_CLASS_INDEX] *= MEL_CLASS_WEIGHT_MULTIPLIER
class_loss_weights = class_loss_weights.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_loss_weights)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)
print(f"Using device: {DEVICE}")
print(f"Class loss weights: {class_loss_weights}")

Using device: cuda
Class loss weights: tensor([2.0000, 2.0000, 1.3021, 2.0000, 1.9291, 0.5000, 2.0000],
       device='cuda:0')


In [10]:
# Training + validation
num_epochs = EPOCHS
best_preds = None
best_labels = None
best_melanoma_recall = 0.0

# Auto-increment checkpoint versioning
i = 1
while (PROJECT_ROOT / f"outputs/checkpoints/best_melanoma_recall_efficientnet_b3_v{i}.pt").exists():
    i += 1
best_checkpoint_path = PROJECT_ROOT / f"outputs/checkpoints/best_melanoma_recall_efficientnet_b3_v{i}.pt"
best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    # Unfreeze backbone at UNFREEZE_EPOCH with differential lr
    if epoch == UNFREEZE_EPOCH:
        model = build_model(unfreeze_backbone=True).to(DEVICE)
        if best_checkpoint_path.exists():
            model.load_state_dict(torch.load(best_checkpoint_path, weights_only=True))
            print(f"Epoch {epoch+1}: loaded best checkpoint for unfreeze")
        else:
            print(f"Epoch {epoch+1}: no checkpoint found, unfreezing from current weights")
        optimizer = torch.optim.Adam([
            {"params": model.features.parameters(), "lr": LEARNING_RATE * 0.1},
            {"params": model.classifier.parameters(), "lr": LEARNING_RATE}
        ])

    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).long()
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).long()
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    melanoma_recall = recall_score(
        all_labels, all_preds, labels=[MEL_CLASS_INDEX], average="macro", zero_division=0
    )

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"Macro Recall: {macro_recall:.4f} | "
        f"Melanoma Recall: {melanoma_recall:.4f}"
    )

    if melanoma_recall > best_melanoma_recall:
        best_melanoma_recall = melanoma_recall
        best_preds = all_preds.copy()
        best_labels = all_labels.copy()
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"Saved new best melanoma recall checkpoint: {best_checkpoint_path}")

Epoch 1/20 | Loss: 1.8395 | Accuracy: 0.0954 | Macro Recall: 0.3552 | Melanoma Recall: 0.4484
Saved new best melanoma recall checkpoint: /workspace/skincancer-pytorch/outputs/checkpoints/best_melanoma_recall_efficientnet_b3_v3.pt
Epoch 2/20 | Loss: 1.6910 | Accuracy: 0.1183 | Macro Recall: 0.4268 | Melanoma Recall: 0.4798
Saved new best melanoma recall checkpoint: /workspace/skincancer-pytorch/outputs/checkpoints/best_melanoma_recall_efficientnet_b3_v3.pt
Epoch 3/20 | Loss: 1.5868 | Accuracy: 0.1373 | Macro Recall: 0.4475 | Melanoma Recall: 0.5919
Saved new best melanoma recall checkpoint: /workspace/skincancer-pytorch/outputs/checkpoints/best_melanoma_recall_efficientnet_b3_v3.pt
Epoch 4/20 | Loss: 1.5164 | Accuracy: 0.1433 | Macro Recall: 0.4624 | Melanoma Recall: 0.5336
Epoch 5/20 | Loss: 1.4604 | Accuracy: 0.1692 | Macro Recall: 0.4893 | Melanoma Recall: 0.5381
Epoch 6/20 | Loss: 1.4124 | Accuracy: 0.1917 | Macro Recall: 0.4785 | Melanoma Recall: 0.5740
Epoch 7/20 | Loss: 1.3555 | 

In [ ]:
print(f"Using device: {DEVICE}")
model = model.to(DEVICE)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)

# melanoma class = 4 based on our label map
melanoma_recall = recall_score(
    all_labels,
    all_preds,
    labels=[4],
    average="macro",
    zero_division=0
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Melanoma Recall: {melanoma_recall:.4f}")

Using device: cuda


In [ ]:
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
print(report)

# Auto-increment report filename to match checkpoint versioning
i = 1
while (PROJECT_ROOT / f"outputs/reports/classification_report_v{i}.txt").exists():
    i += 1

report_path = PROJECT_ROOT / f"outputs/reports/classification_report_v{i}.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)

elapsed = time.time() - start_time

with open(report_path, "w") as f:
    f.write(f"EPOCHS: {EPOCHS} | SAMPLE_FRAC: {SAMPLE_FRAC}\n")
    f.write(f"Total runtime: {elapsed/60:.1f} minutes\n\n")
    f.write(report)

print(f"Saved report to: {report_path}")

In [ ]:
best_report = classification_report(best_labels, best_preds, target_names=CLASS_NAMES)
print("Best Model Classification Report")
print(best_report)

# Match versioning to checkpoint
i = 1
while (PROJECT_ROOT / f"outputs/reports/best_classification_report_v{i}.txt").exists():
    i += 1

best_report_path = PROJECT_ROOT / f"outputs/reports/best_classification_report_v{i}.txt"
best_report_path.parent.mkdir(parents=True, exist_ok=True)

with open(best_report_path, "w") as f:
    f.write(f"EPOCHS: {EPOCHS} | SAMPLE_FRAC: {SAMPLE_FRAC}\n")    
    f.write("Best Model Classification Report\n\n")
    f.write(best_report)

print(f"Saved report to: {best_report_path}")

In [ ]:
checkpoint_dir = PROJECT_ROOT / "outputs/checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Auto-increment filename
i = 1
while (checkpoint_dir / f"local_poc_mobilenetv3_small_v{i}.pt").exists():
    i += 1

checkpoint_path = checkpoint_dir / f"local_poc_mobilenetv3_small_v{i}.pt"

torch.save(model.state_dict(), checkpoint_path)
print(f"Saved checkpoint to: {checkpoint_path}")



In [ ]:
# Reload best checkpoint and verify
print(f"Loading from: {best_checkpoint_path}")
model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).long()
        preds = torch.argmax(model(images), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

reloaded_melanoma_recall = recall_score(all_labels, all_preds, labels=[4], average="macro", zero_division=0)
print(f"Expected (best training recall): {best_melanoma_recall:.4f}")
print(f"Reloaded checkpoint:             {reloaded_melanoma_recall:.4f}")
print(f"Match: {abs(reloaded_melanoma_recall - best_melanoma_recall) < 0.0001}")

In [ ]:
elapsed = time.time() - start_time
print(f"Total runtime: {elapsed/60:.1f} minutes")

In [ ]:
print("Training complete. Stop the pod in RunPod dashboard to end billing.")

In [ ]:
# Check class distribution in a batch
images, labels = next(iter(train_loader))
print(labels)
print(labels.unique(return_counts=True))

In [ ]:
print(class_loss_weights)

In [ ]:
print(MEL_CLASS_INDEX)